# NfgTransformer SRE Solver Training

This notebook trains and evaluates a neural SRE solver for the normal-form games produced by Level-Based Foraging stage games.

Workflow:

1. Train an NfgTransformer checkpoint on synthetic LBF-like N-player games.
2. Load the latest saved checkpoint, which can be a partially trained interrupted run.
3. Generate held-out random normal-form games.
4. Evaluate the checkpoint.

Use `discrete_action_space/lbf_grid/deep_srq_nplayer_ablation.ipynb` to run Level-Based Foraging with the trained checkpoint.

In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
while REPO_ROOT.name != 'SRE-DQN' and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

PACKAGE_ROOT = REPO_ROOT / 'discrete_action_space' / 'sre_solvers' / 'nfg_transformer'
DATA_ROOT = PACKAGE_ROOT / 'nfg_sre_data'
CKPT_ROOT = PACKAGE_ROOT / 'nfg_sre_checkpoints'

GAME_SHAPES = ((6, 6), (6, 6, 6))
EVAL_GAME_SHAPE = '6x6x6'
VAL_DIR = DATA_ROOT / 'val_lbf3_random'
CHECKPOINT = CKPT_ROOT / 'nfg_sre_lbf3_online.pt'

VAL_SAMPLES = 500
SHARD_SIZE = 500

DATA_ROOT.mkdir(parents=True, exist_ok=True)
CKPT_ROOT.mkdir(parents=True, exist_ok=True)
print(REPO_ROOT)

/home/wowthecoder/SRE-DQN


## 1. Train the NfgTransformer checkpoint

In [ ]:
from discrete_action_space.sre_solvers.nfg_transformer.train import train_checkpoint

train_checkpoint(
    output=CHECKPOINT,
    num_iterations=20_000,  # Match the reference notebook update budget. Use 500 for a quick smoke run.
    log_every=1_000,
    batch_size=128,
    lr=3e-4,
    embed_dim=64,
    num_blocks=8,
    num_heads=8,
    num_self_attend_per_block=1,
    game_shapes=GAME_SHAPES,
    seed=2025,
    use_gpu=True,
)

training_mode=online_synthetic_robust_gap shapes=((6, 6), (6, 6, 6)) batch_size=128 iterations=20000 device=cuda


nfg-sre-train:   5%|█▊                                  | 1000/20000 [17:14<3:59:35,  1.32it/s, gap=0.1292, loss=0.1292]

iteration=1000 loss=0.129233 robust_gap=0.129233


nfg-sre-train:  10%|███▌                                | 2000/20000 [34:36<3:17:08,  1.52it/s, gap=0.0769, loss=0.0769]

iteration=2000 loss=0.076889 robust_gap=0.076889


...

nfg-sre-train:  80%|██████████████████████████▍      | 16000/20000 [4:43:43<1:11:21,  1.07s/it, gap=0.0540, loss=0.0540]

iteration=16000 loss=0.054048 robust_gap=0.054048


nfg-sre-train:  85%|█████████████████████████████▊     | 17000/20000 [5:01:21<48:15,  1.04it/s, gap=0.0556, loss=0.0556]

iteration=17000 loss=0.055650 robust_gap=0.055650


nfg-sre-train:  85%|████████████████████████████▏    | 17057/20000 [5:02:21<1:12:30,  1.48s/it, gap=0.0556, loss=0.0556]

## 2. Load an interrupted checkpoint

In [2]:
from discrete_action_space.sre_solvers.nfg_transformer.solver import NfgTransformerSreSolver

CHECKPOINT = CKPT_ROOT / 'nfg_sre_lbf3_online.pt'
if not CHECKPOINT.exists():
    raise FileNotFoundError(f'Checkpoint not found: {CHECKPOINT}')

probe_solver = NfgTransformerSreSolver(
    checkpoint_path=CHECKPOINT,
    device='cpu',
    fallback_enabled=False,
)
probe_solver.close()
print(f'Loaded checkpoint: {CHECKPOINT}')


Loaded checkpoint: /home/wowthecoder/SRE-DQN/discrete_action_space/sre_solvers/nfg_transformer/nfg_sre_checkpoints/nfg_sre_lbf3_online.pt


## 3. Generate held-out evaluation data

Evaluation shards use one concrete tensor shape because NumPy arrays cannot mix rectangular game sizes inside one shard.

In [3]:
from discrete_action_space.sre_solvers.nfg_transformer.generate_dataset import generate_dataset

generate_dataset(
    output=VAL_DIR,
    num_samples=VAL_SAMPLES,
    shard_size=SHARD_SIZE,
    num_players=3,
    num_actions=6,
    game_shape=EVAL_GAME_SHAPE,
    seed=2026,
    label_mode='random',
    exploitability_tol=1e-4,
)

nfg-sre-random: 100%|██████████| 500/500 [00:00<00:00, 5950.80it/s]


## 4. Evaluate the checkpoint

The main number is robust exploitability. `accept_rate` uses the strict `exploitability_tol` below; with the short default training run it can be 0 even while the model is useful as a warm start because the integrated solver falls back to PATH.

In [4]:
from discrete_action_space.sre_solvers.nfg_transformer.evaluate import evaluate_checkpoint

evaluate_checkpoint(
    checkpoint=CHECKPOINT,
    data_dir=VAL_DIR,
    exploitability_tol=1e-3,
    device=None,
)

checkpoint=/home/wowthecoder/SRE-DQN/discrete_action_space/sre_solvers/nfg_transformer/nfg_sre_checkpoints/nfg_sre_lbf3_online.pt
samples=500
mean_gap=0.061684
p95_gap=0.229828
accept_tol=0.001000
accept_rate=0.3300
accept_rate@0.1=0.7460
accept_rate@0.05=0.6000
accept_rate@0.01=0.4020
